## 0. Imports

In [64]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix

# Day 09. Exercise 00
# Regularization

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [65]:
df = pd.read_csv('../data/dayofweek.csv', sep=',')

df.head()

,numTrials,hour,dayofweek,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,-0.788667,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.756764,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.724861,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.692958,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.661055,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [66]:
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

In [67]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

- KFold просто делит данные на k частей.

- StratifiedKFold делает то же самое, но сохраняет пропорцию классов

In [68]:
%%time

model = LogisticRegression(random_state=21, fit_intercept=False)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

    model.fit(X_tr, y_tr)
    y_pred_train = model.predict(X_tr)
    y_pred_valid = model.predict(X_val)

    train_acc = accuracy_score(y_tr, y_pred_train)
    valid_acc = accuracy_score(y_val, y_pred_valid)

    train_scores.append(train_acc)
    valid_scores.append(valid_acc)

    print(f"train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}")

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

train -  0.64056   |   valid -  0.65926
train -  0.63561   |   valid -  0.62222
train -  0.64468   |   valid -  0.60000
train -  0.64056   |   valid -  0.64444
train -  0.65375   |   valid -  0.60741
train -  0.62902   |   valid -  0.60000
train -  0.66117   |   valid -  0.60000
train -  0.63726   |   valid -  0.54074
train -  0.63756   |   valid -  0.66418
train -  0.64745   |   valid -  0.61194
Average accuracy on crossval is 0.61502
Std is 0.03399
CPU times: total: 578 ms
Wall time: 612 ms


Кросс-валидация даёт более надёжную оценку качества, усреднённую по 10 независимым проверкам.
-  берут np.mean(valid_scores) — это «средняя точность модели в реальной жизни».
-  np.std(valid_scores) — показывает, насколько результаты «скачут» между разными разбиениями (низкое std — значит модель стабильна).

1. Что показывает train_acc

- Это точность на тех данных, на которых модель училась.
- Если модель «зазубрила» данные, train_acc будет очень высоким (часто 0.99+).

2. Что показывает valid_acc

- Это точность на новых данных (валидационных).
- Если valid_acc сильно ниже, чем train_acc, значит модель переобучилась — выучила шум, а не закономерности.

### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [69]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

In [70]:
combinations = [
    (None, 'lbfgs'),     # без регуляризации
    ('l2', 'lbfgs'),      # классический Ridge
    ('l1', 'liblinear'),  # Lasso
    ('l2', 'liblinear')  # Ridge с другим решателем
]

penalty — это тип регуляризации, то есть способ «наказать» модель за слишком большие коэффициенты, чтобы не переобучалась.
- none — без регуляризации (модель может переобучиться).
- l1 — Lasso: зануляет некоторые коэффициенты, делает модель разреженной (важные признаки остаются).
- l2 — Ridge: «сжимает» коэффициенты, но не зануляет (все признаки сохраняются, просто становятся меньше).

solver — это алгоритм оптимизации, который решает задачу минимизации функции потерь.
- 'lbfgs' — универсальный, быстрый, но не работает с l1.
- 'liblinear' — старый, умеет и l1, и l2.

In [71]:
results = []

for penalty, solver in combinations:
    model = LogisticRegression(
        penalty=penalty,
        solver=solver,
        random_state=21,
        fit_intercept=False,
        max_iter=1000
    )
    
    train_scores = []
    valid_scores = []

    for train_idx, valid_idx in skf.split(X_train, y_train):
        X_t, X_v = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_t, y_v = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        model.fit(X_t, y_t)
        train_acc = model.score(X_t, y_t)
        valid_acc = model.score(X_v, y_v)

        train_scores.append(train_acc)
        valid_scores.append(valid_acc)
        print(f"penalty={penalty}, solver={solver} | train - {train_acc:.5f} | valid - {valid_acc:.5f}")

    avg_acc = np.mean(valid_scores)
    std_acc = np.std(valid_scores)
    print(f"Average accuracy on crossval is {avg_acc:.5f}")
    print(f"Std is {std_acc:.5f}")

    results.append({
        'penalty': penalty,
        'solver': solver,
        'avg_acc': avg_acc,
        'std_acc': std_acc
    })

penalty=None, solver=lbfgs | train - 0.65952 | valid - 0.67407
penalty=None, solver=lbfgs | train - 0.67189 | valid - 0.60741
penalty=None, solver=lbfgs | train - 0.65787 | valid - 0.62963
penalty=None, solver=lbfgs | train - 0.66035 | valid - 0.66667
penalty=None, solver=lbfgs | train - 0.66777 | valid - 0.60741
penalty=None, solver=lbfgs | train - 0.64716 | valid - 0.62963
penalty=None, solver=lbfgs | train - 0.66859 | valid - 0.59259
penalty=None, solver=lbfgs | train - 0.66447 | valid - 0.60741
penalty=None, solver=lbfgs | train - 0.65980 | valid - 0.70149
penalty=None, solver=lbfgs | train - 0.67051 | valid - 0.62687
Average accuracy on crossval is 0.63432
Std is 0.03340
penalty=l2, solver=lbfgs | train - 0.64056 | valid - 0.65926
penalty=l2, solver=lbfgs | train - 0.63561 | valid - 0.62222
penalty=l2, solver=lbfgs | train - 0.64468 | valid - 0.60000
penalty=l2, solver=lbfgs | train - 0.64056 | valid - 0.64444
penalty=l2, solver=lbfgs | train - 0.65375 | valid - 0.60741
penalty=l2

c:\Users\Анастасия\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
c:\Users\Анастасия\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
c:\Users\Анастасия\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classifica

penalty=l1, solver=liblinear | train - 0.63397 | valid - 0.60000
penalty=l1, solver=liblinear | train - 0.61913 | valid - 0.59259
penalty=l1, solver=liblinear | train - 0.63974 | valid - 0.59259
penalty=l1, solver=liblinear | train - 0.62737 | valid - 0.52593
penalty=l1, solver=liblinear | train - 0.61367 | valid - 0.66418
penalty=l1, solver=liblinear | train - 0.62768 | valid - 0.57463
Average accuracy on crossval is 0.60240
Std is 0.03627
penalty=l2, solver=liblinear | train - 0.60758 | valid - 0.62222
penalty=l2, solver=liblinear | train - 0.61336 | valid - 0.59259


c:\Users\Анастасия\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
c:\Users\Анастасия\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
c:\Users\Анастасия\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classifica

penalty=l2, solver=liblinear | train - 0.61583 | valid - 0.58519
penalty=l2, solver=liblinear | train - 0.62077 | valid - 0.60741
penalty=l2, solver=liblinear | train - 0.62902 | valid - 0.59259
penalty=l2, solver=liblinear | train - 0.61171 | valid - 0.58519
penalty=l2, solver=liblinear | train - 0.64303 | valid - 0.60000
penalty=l2, solver=liblinear | train - 0.61830 | valid - 0.55556
penalty=l2, solver=liblinear | train - 0.62438 | valid - 0.66418
penalty=l2, solver=liblinear | train - 0.61944 | valid - 0.58955
Average accuracy on crossval is 0.59945
Std is 0.02701


c:\Users\Анастасия\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
c:\Users\Анастасия\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
c:\Users\Анастасия\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classifica

In [72]:
results_df = pd.DataFrame(results)
results_df.sort_values(by='avg_acc', ascending=False)

,penalty,solver,avg_acc,std_acc
0,None,lbfgs,0.634317,0.033395
1,l2,lbfgs,0.615019,0.033990
2,l1,liblinear,0.602399,0.036270
3,l2,liblinear,0.599447,0.027014


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [73]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

model = SVC(probability=True, kernel='linear', random_state=21)

train_scores = []
valid_scores = []

for train_idx, valid_idx in skf.split(X_train, y_train):
    X_t, X_v = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_t, y_v = y_train.iloc[train_idx], y_train.iloc[valid_idx]

    model.fit(X_t, y_t)
    train_acc = model.score(X_t, y_t)
    valid_acc = model.score(X_v, y_v)

    train_scores.append(train_acc)
    valid_scores.append(valid_acc)

    print(f"train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}")

avg_acc = np.mean(valid_scores)
std_acc = np.std(valid_scores)

print(f"Average accuracy on crossval is {avg_acc:.5f}")
print(f"Std is {std_acc:.5f}")

train -  0.70651   |   valid -  0.68148
train -  0.68920   |   valid -  0.64444
train -  0.69744   |   valid -  0.66667
train -  0.68920   |   valid -  0.65926
train -  0.69497   |   valid -  0.63704
train -  0.68673   |   valid -  0.68148
train -  0.69827   |   valid -  0.61481
train -  0.70486   |   valid -  0.57778
train -  0.68863   |   valid -  0.72388
train -  0.71005   |   valid -  0.64179
Average accuracy on crossval is 0.65286
Std is 0.03800


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [74]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
results = []

In [75]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

In [76]:
for C in C_values:
    model = SVC(probability=True, random_state=21, kernel='linear', C=C)
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='accuracy',n_jobs=-1)
    results.append((C, scores.mean(), scores.std()))
    print(f'C - {C:<7}  |  mean accuracy = {scores.mean():.5f}  |  std = {scores.std():.5f}')

best = max(results, key=lambda x: x[1])
print("\nBest C:", best[0])
print(f"Average accuracy on crossval is {best[1]:.5f}")
print(f"Std is {best[2]:.5f}")


C - 0.001    |  mean accuracy = 0.23443  |  std = 0.00397
C - 0.01     |  mean accuracy = 0.38125  |  std = 0.02293
C - 0.1      |  mean accuracy = 0.55787  |  std = 0.02522
C - 1        |  mean accuracy = 0.65286  |  std = 0.03800
C - 10       |  mean accuracy = 0.71814  |  std = 0.04026
C - 100      |  mean accuracy = 0.75001  |  std = 0.02849

Best C: 100
Average accuracy on crossval is 0.75001
Std is 0.02849


- cross_val_score — автоматическая кросс-валидация: сама делит данные на фолды, обучает и возвращает метрики. когда нужна только средняя точность.

- Цикл с StratifiedKFold — ручной способ: контроль над процессом, позволяет вывод точности по фолдам

## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [77]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_tr, y_val = y_train.iloc[train_index], y_train.iloc[valid_index]

    model = DecisionTreeClassifier(max_depth=10, random_state=21)
    model.fit(X_tr, y_tr)

    y_pred_train = model.predict(X_tr)
    y_pred_valid = model.predict(X_val)

    acc_train = accuracy_score(y_tr, y_pred_train)
    acc_valid = accuracy_score(y_val, y_pred_valid)

    train_scores.append(acc_train)
    valid_scores.append(acc_valid)

    print(f"train -  {acc_train:.5f}   |   valid -  {acc_valid:.5f}")

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

train -  0.80874   |   valid -  0.77037
train -  0.79802   |   valid -  0.70370
train -  0.81286   |   valid -  0.72593
train -  0.80049   |   valid -  0.74815
train -  0.80956   |   valid -  0.68889
train -  0.78978   |   valid -  0.74074
train -  0.80627   |   valid -  0.60741
train -  0.82688   |   valid -  0.71111
train -  0.78995   |   valid -  0.79104
train -  0.80313   |   valid -  0.70896
Average accuracy on crossval is 0.71963
Std is 0.04791


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [78]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

In [79]:
max_depth_values = [3, 5, 7, 10, 15, 20, None] 
criterion_values = ["gini", "entropy"]
class_weight_values = [None, "balanced"]

In [80]:
results = []

for depth in max_depth_values:
    for crit in criterion_values:
        for weight in class_weight_values:
            train_scores = []
            valid_scores = []

            for train_index, valid_index in skf.split(X_train, y_train):
                X_tr, X_val = X_train.iloc[train_index], X_train.iloc[valid_index]
                y_tr, y_val = y_train.iloc[train_index], y_train.iloc[valid_index]

                model = DecisionTreeClassifier(
                    max_depth=depth,
                    criterion=crit,
                    class_weight=weight,
                    random_state=21
                )
                model.fit(X_tr, y_tr)

                y_pred_train = model.predict(X_tr)
                y_pred_valid = model.predict(X_val)

                acc_train = accuracy_score(y_tr, y_pred_train)
                acc_valid = accuracy_score(y_val, y_pred_valid)

                train_scores.append(acc_train)
                valid_scores.append(acc_valid)

            results.append({
                "max_depth": depth,
                "criterion": crit,
                "class_weight": weight,
                "mean_train_acc": np.mean(train_scores),
                "mean_valid_acc": np.mean(valid_scores),
                "std_valid_acc": np.std(valid_scores)
            })

In [81]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="mean_valid_acc", ascending=False)
df_results.reset_index(drop=True, inplace=True)

In [82]:
df_results.head(10)

,max_depth,criterion,class_weight,mean_train_acc,mean_valid_acc,std_valid_acc
0,20.0,gini,None,0.988956,0.887977,0.018618
1,NaN,gini,None,1.000000,0.887966,0.018397
2,NaN,entropy,None,1.000000,0.887253,0.026070
3,NaN,entropy,balanced,1.000000,0.882067,0.026125
4,20.0,entropy,balanced,0.996290,0.879851,0.025108
5,NaN,gini,balanced,1.000000,0.877590,0.022566
6,20.0,gini,balanced,0.991427,0.875368,0.023406
7,20.0,entropy,None,0.987554,0.873134,0.025028
8,15.0,entropy,balanced,0.965546,0.859071,0.028024
9,15.0,gini,None,0.946258,0.856821,0.027801


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [83]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_index, valid_index in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_tr, y_val = y_train.iloc[train_index], y_train.iloc[valid_index]

    model = RandomForestClassifier(
        n_estimators=50,
        max_depth=14,
        random_state=21
    )
    model.fit(X_tr, y_tr)

    y_pred_train = model.predict(X_tr)
    y_pred_valid = model.predict(X_val)

    acc_train = accuracy_score(y_tr, y_pred_train)
    acc_valid = accuracy_score(y_val, y_pred_valid)

    print(f"train -  {acc_train:.5f}   |   valid -  {acc_valid:.5f}")

    train_scores.append(acc_train)
    valid_scores.append(acc_valid)

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")


train -  0.97939   |   valid -  0.85185
train -  0.96620   |   valid -  0.85926
train -  0.96208   |   valid -  0.91852
train -  0.97115   |   valid -  0.91852
train -  0.97197   |   valid -  0.88148
train -  0.96538   |   valid -  0.86667
train -  0.96455   |   valid -  0.88889
train -  0.96867   |   valid -  0.87407
train -  0.96458   |   valid -  0.93284
train -  0.96787   |   valid -  0.86567
Average accuracy on crossval is 0.88578
Std is 0.02673


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [84]:
max_depth_values = [5, 10, 14, 20, 25, None]
n_estimators_values = [10, 50, 100, 200]
class_weight_values = [None, "balanced"]

In [ ]:
results = []

for depth in max_depth_values:
    for n_est in n_estimators_values:
        for weight in class_weight_values:
            train_scores = []
            valid_scores = []

            for train_index, valid_index in skf.split(X_train, y_train):
                X_tr, X_val = X_train.iloc[train_index], X_train.iloc[valid_index]
                y_tr, y_val = y_train.iloc[train_index], y_train.iloc[valid_index]

                model = RandomForestClassifier(
                    n_estimators=n_est,
                    max_depth=depth,
                    class_weight=weight,
                    random_state=21,
                    n_jobs=-1
                )
                model.fit(X_tr, y_tr)

                y_pred_train = model.predict(X_tr)
                y_pred_valid = model.predict(X_val)

                acc_train = accuracy_score(y_tr, y_pred_train)
                acc_valid = accuracy_score(y_val, y_pred_valid)

                train_scores.append(acc_train)
                valid_scores.append(acc_valid)

            results.append({
                "n_estimators": n_est,
                "max_depth": depth,
                "class_weight": weight,
                "mean_train_acc": np.mean(train_scores),
                "mean_valid_acc": np.mean(valid_scores),
                "std_valid_acc": np.std(valid_scores)
            })


In [ ]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="mean_valid_acc", ascending=False)
df_results.reset_index(drop=True, inplace=True)

In [ ]:
df_results.head(10)

,n_estimators,max_depth,class_weight,mean_train_acc,mean_valid_acc,std_valid_acc
0,200,NaN,balanced,1.000000,0.925553,0.019342
1,100,25.0,balanced,1.000000,0.924891,0.020045
2,200,25.0,None,1.000000,0.924882,0.021352
3,200,NaN,None,1.000000,0.924224,0.020741
4,200,20.0,balanced,0.999634,0.923575,0.022683
5,200,25.0,balanced,1.000000,0.923571,0.021332
6,100,25.0,None,0.999927,0.923567,0.018048
7,100,NaN,balanced,1.000000,0.922917,0.020641
8,100,20.0,balanced,0.999195,0.922917,0.019576
9,100,NaN,None,1.000000,0.922255,0.019308


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [ ]:
best_model = RandomForestClassifier(n_estimators=200, max_depth=None, class_weight='balanced', random_state=21)

In [ ]:
best_model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
y_pred = best_model.predict(X_test)

In [ ]:
final_acc = accuracy_score(y_pred, y_test)
final_acc

0.9822485207100592

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=np.unique(y_test), columns=np.unique(y_test))

cm_df

,0,1,2,3,4,5,6
0,26,0,0,0,0,0,1
1,0,54,1,0,0,0,0
2,0,0,29,1,0,0,0
3,0,0,0,80,0,0,0
4,0,0,0,0,20,0,1
5,0,0,0,0,0,53,1
6,0,0,0,0,0,1,70


In [ ]:
cm

array([[26,  0,  0,  0,  0,  0,  1],
       [ 0, 54,  1,  0,  0,  0,  0],
       [ 0,  0, 29,  1,  0,  0,  0],
       [ 0,  0,  0, 80,  0,  0,  0],
       [ 0,  0,  0,  0, 20,  0,  1],
       [ 0,  0,  0,  0,  0, 53,  1],
       [ 0,  0,  0,  0,  0,  1, 70]])

In [ ]:
errors = (cm_df.sum(axis=1) - np.diag(cm_df))  # ошибки по каждому классу
total = cm_df.sum(axis=1)
error_pct = (errors / total * 100).round(2)

In [ ]:
error_analysis = pd.DataFrame({
    "weekday": cm_df.index,
    "error_%": error_pct
}).sort_values("error_%", ascending=False)

In [ ]:
error_analysis

,weekday,error_%
4,4,4.76
0,0,3.70
2,2,3.33
5,5,1.85
1,1,1.82
6,6,1.41
3,3,0.00


In [ ]:
joblib.dump(best_model, "../data/best_random_forest_model.pkl")

['../data/best_random_forest_model.pkl']